<a href="https://colab.research.google.com/github/Yuva2832/FMML_LAB_PROJECT/blob/main/Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **FOUNDATIONS OF MODERN MACHINE LEARNING, IIIT Hyderabad**
### MODULE: CLASSIFICATION-1
### LAB-3 : Using KNN for Text Classification
#### Module Coordinator: Jashn Arora


---

## **Section 1: Understanding NLP tools**

In this lab we will be using KNN on a real world NLP application i.e. is text classification. But first look at some NLP techniques for text classification and tools that we use when we want to use python for NLP.

## Section 1.2: Data Cleaning and Preprocessing step

Raw text must be processed and converted into a form so that it is suitable to use with various machine-learning algorithms.  
In case of text, there are lots of things that need to be taken into account.  


1.   Removing numbers from the text
2.   Handling capitalization and punctuation.
3.   Stemming and Lemmatizing text.  

And most importantly, one can't just use words or images directly in algorithms; they need to be converted into vectors- a form that algorithms can understand.



### **NLTK**
NLTK (or Natural Language Tool Kit) is a commonly used library for processing text. We will use this tool in this lab. Lets first install it.


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
import re
import numpy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from bs4 import BeautifulSoup

def cleanText(text, lemmatize, stemmer):
    """Method for cleaning text from train and test data. Removes numbers, punctuation, and capitalization. Stems or lemmatizes text."""

    if isinstance(text, float):
        text = str(text)
    if isinstance(text, numpy.int64):
        text = str(text)
    try:
        text = text.decode()
    except AttributeError:
        pass

    soup = BeautifulSoup(text, "lxml")
    text = soup.get_text()
    text = re.sub(r"[^A-Za-z]", " ", text)
    text = text.lower()


    if lemmatize:
        wordnet_lemmatizer = WordNetLemmatizer()

        def get_tag(tag):
            if tag.startswith('J'):
                return wordnet.ADJ
            elif tag.startswith('V'):
                return wordnet.VERB
            elif tag.startswith('N'):
                return wordnet.NOUN
            elif tag.startswith('R'):
                return wordnet.ADV
            else:
                return ''

        text_result = []
        tokens = word_tokenize(text)  # Generate list of tokens
        tagged = pos_tag(tokens)
        for t in tagged:
            try:
                text_result.append(wordnet_lemmatizer.lemmatize(t[0], get_tag(t[1][:2])))
            except:
                text_result.append(wordnet_lemmatizer.lemmatize(t[0]))
        return text_result

    if stemmer:
        text_result = []
        tokens = word_tokenize(text)
        snowball_stemmer = SnowballStemmer('english')
        for t in tokens:
            text_result.append(snowball_stemmer.stem(t))
        return text_result

In [ ]:
sample_text = "Troubling"
sample_text_result = cleanText(sample_text, lemmatize=False, stemmer=True)
sample_text_result = " ".join(str(x) for x in sample_text_result)
print(sample_text)
print(sample_text_result)
sample_text_result = cleanText(sample_text, lemmatize=True, stemmer=False)
sample_text_result = " ".join(str(x) for x in sample_text_result)
print(sample_text_result)

## Section 1.2: BAG OF WORDS

A bag-of-words model, or BoW for short, is a way of extracting features from text for use in modeling, such as with machine learning algorithms.

The approach is very simple and flexible, and can be used in many ways for extracting features from documents.

A bag-of-words is a representation of text that describes the occurrence of words within a document.
It is called a ?bag? of words, because any information about the order or structure of words in the document is discarded. The model is only concerned with whether known words occur in the document, not where in the document.

In [ ]:
# Functions to convert document(s) to a list of words, with the option of removing stopwords. Returns document-term matrix.

def createBagOfWords(train, test, remove_stopwords, lemmatize, stemmer):
    if remove_stopwords:
        vectorizer = CountVectorizer(analyzer='word', input='content', stop_words=stopwords.words('english'))
    else:
        vectorizer = CountVectorizer(analyzer='word', input='content')

    clean_train = []
    for paragraph in train:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_train.append(paragraph)

    clean_test = []
    for paragraph in test:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_test.append(paragraph)

    bag_of_words_train = vectorizer.fit_transform(clean_train).toarray()
    bag_of_words_test = vectorizer.transform(clean_test).toarray()
    return bag_of_words_train, bag_of_words_test


## Section 1.3: TF-IDF
TF-IDF technique is used to find meaning of sentences consisting of words and cancels out the incapabilities of Bag of Words technique which is good for text classification or for helping a machine read words in numbers.

The number of times a term occurs in a document is called its Term frequency (TF).

 Document frequency is the number of documents in which the word is present.  Inverse DF (IDF) is the inverse of the document frequency which measures the informativeness of term *t*.




In [ ]:
def createTFIDF(train, test, remove_stopwords, lemmatize, stemmer):
    if remove_stopwords:
        vectorizer = TfidfVectorizer(analyzer='word', input='content', stop_words=stopwords.words('english'))
    else:
        vectorizer =  TfidfVectorizer(analyzer='word', input='content')

    clean_train = []
    for paragraph in train:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_train.append(paragraph)

    clean_test = []
    for paragraph in test:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_test.append(paragraph)

    tfidf_train = vectorizer.fit_transform(clean_train).toarray()
    tfidf_test = vectorizer.transform(clean_test).toarray()
    return tfidf_train, tfidf_test

# **Section 2: UNDERSTANDING THE DATA : A REVIEWS DATASET**

Sentiment analysis is the interpretation and classification of emotions (such as positive, negative and neutral) within text data using text analysis techniques.  
Given below is a dataset consisting of reviews along with sentiment class (positive or negative).

In [ ]:
# Upload the Reviews CSV file that has been shared with you.
# Run this cell, click on the 'Choose files' button and upload the file.
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv('reviews.csv')

In [ ]:
df = df.dropna()

In [ ]:
df.to_csv('reviews.csv', index=False)

# **Section 3: KNN MODEL**

Given below are two KNN models; in the first case we are using Bag-of-Words and in the second case we are using TF-IDF.
Note the different metrics and parameters used in each.

In [ ]:
from sklearn import metrics, neighbors
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

## TASK - 1: Tweak the models below and see results with different parameters and distance metrics.

def bow_knn():
    """Method for determining nearest neighbors using bag-of-words and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('reviews.csv')
    X_train, X_test, y_train, y_test = train_test_split(training_data["sentence"], training_data["sentiment"], test_size=0.2, random_state=5)
    X_train, X_test = createBagOfWords(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    # print(X_train)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='euclidean', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with BOW accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    print('\n')
    return predicted, y_test


def tfidf_knn():
    """Method for determining nearest neighbors using tf-idf and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('reviews.csv')
    X_train, X_test, y_train, y_test = train_test_split(training_data["sentence"], training_data["sentiment"],
                                                        test_size=0.2, random_state=5)
    X_train, X_test = createTFIDF(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    # print(X_train)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='distance', algorithm='brute', leaf_size=30, p=2,
                                         metric='cosine', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with TFIDF accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    return predicted, y_test

Note: Cross-validation will be discussed in detail in the upcoming lab session.

In [ ]:
## KNN accuracy after using BoW
predicted, y_test = bow_knn()

In [ ]:
## KNN accuracy after using TFIDF
predicted, y_test = tfidf_knn()

# Section 4: SPAM TEXT DATASET
Now let's use what we've learnt to classify texts as spam or not spam.

In [ ]:
# Upload the spam text data CSV file that has been shared with you. You can also download the file from https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
# Run this cell, click on the 'Choose files' button and upload the file.
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv('spam.csv')
df

In [ ]:
df['Category'] = df['Category'].map({'ham': 0, 'spam': 1})

In [ ]:
df.head(5)

In [ ]:
len(df)

In [ ]:
from sklearn import metrics, neighbors
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

## TASK - 2: Tweak the models below and see results with different parameters and distance metrics.

def bow_knn():
    """Method for determining nearest neighbors using bag-of-words and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('spam.csv')
    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})
    X_train, X_test, y_train, y_test = train_test_split(training_data["Message"], training_data["Category"], test_size=0.2, random_state=5)
    X_train, X_test = createBagOfWords(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='euclidean', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with BOW accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    print('\n')
    return predicted, y_test


def tfidf_knn():
    """Method for determining nearest neighbors using tf-idf and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('spam.csv')
    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})
    X_train, X_test, y_train, y_test = train_test_split(training_data["Message"], training_data["Category"], test_size=0.2, random_state=5)
    X_train, X_test = createTFIDF(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='distance', algorithm='brute', leaf_size=30, p=2, metric='cosine', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with TFIDF accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    return predicted, y_test

In [ ]:
# This cell may take some time to run
predicted, y_test = bow_knn()

In [ ]:
# This cell may take some time to run
predicted, y_test = tfidf_knn()

### Questions to Think About and Answer
1. Why does the TF-IDF approach generally result in a better accuracy than Bag-of-Words ?
2. Can you think of techniques that are better than both BoW and TF-IDF ?
3. Read about Stemming and Lemmatization from the resources given below. Think about the pros/cons of each.

The TF-IDF (Term Frequency-Inverse Document Frequency) approach generally results in better accuracy than Bag-of-Words (BoW) because it incorporates both the local importance of a term within a document and its global importance across the entire corpus. This provides more meaningful features for machine learning algorithms to use. Here's why TF-IDF typically outperforms BoW:


---

1. Reduces the Impact of Common Words

BoW represents text simply as a collection of word counts, without accounting for whether a word is common across all documents (e.g., "the", "is", "in"). Words that occur frequently across many documents are given the same weight as more meaningful terms.

TF-IDF, on the other hand, assigns lower weights to words that appear frequently across the corpus (i.e., common words), and higher weights to words that appear frequently in a single document but not across many documents. This means that rare words which are more informative are given more weight, improving accuracy.

This helps TF-IDF to focus on terms that are more likely to provide unique information about a document's content, which enhances classification accuracy.


2. Highlights Relevant Features

BoW simply counts the frequency of each word, which can result in high-dimensional vectors with redundant information. Without considering the importance of words in context, BoW might misrepresent the relationship between terms.

TF-IDF gives higher importance to terms that are specific to a given document rather than those that are commonly found across many documents. This helps to focus the model's attention on more meaningful, distinctive words, especially in tasks like document classification or information retrieval.

For example, in a classification task, terms that are unique to a class are weighted more heavily in TF-IDF, allowing the model to better differentiate between classes.


3. Handles Rare Words Better

BoW can give overly simplistic representations for rare terms by counting their presence in documents, leading to their underrepresentation or incorrect classification. In contrast, TF-IDF adjusts for this by factoring in inverse document frequency (IDF), which increases the importance of rare terms that may be highly relevant to specific topics.

This makes TF-IDF particularly beneficial in tasks like topic modeling, where capturing the relevance of rare or niche terms is crucial.


4. Better Handling of High-Dimensional Data

BoW often results in sparse and high-dimensional data, which can lead to overfitting and poor generalization. By focusing on more informative features through TF-IDF, the model is likely to perform better in high-dimensional spaces, reducing noise and improving generalization.



---

Studies and Evidence

Research has shown that TF-IDF often yields better performance than BoW in tasks such as document classification, sentiment analysis, and search engine ranking (Joachims, 1998). In the case of text classification, TF-IDF's ability to weight terms based on their informativeness allows classifiers to distinguish between documents more effectively than BoW, which treats all words equally.

A comparison by Ramos (2003) on text classification tasks demonstrated that TF-IDF improves classification accuracy by better representing the significance of words in a document, compared to BoW's raw frequency counts.



---

Conclusion

In summary, TF-IDF improves upon BoW by reducing the impact of common words, emphasizing rare or distinctive words, and focusing on more informative features that are better at capturing the unique content of documents. As a result, TF-IDF leads to more accurate and meaningful representations for machine learning tasks.

Sources:

Joachims, T. (1998). Text Classification with Support Vector Machines: Learning with Many Relevant Features. European Conference on Machine Learning (ECML).

Ramos, J. (2003). Using TF-IDF to Determine Word Relevance in Document Queries. Information Retrieval.

Yes, there are several advanced techniques that can improve upon BoW and TF-IDF for text representation. Here are a few methods:


---

1. Word Embeddings (e.g., Word2Vec, GloVe)

How it works: Word embeddings, such as Word2Vec (Mikolov et al., 2013) and GloVe (Pennington et al., 2014), convert words into dense, continuous vector representations, where semantically similar words are represented by similar vectors.

Advantage over BoW/TF-IDF:

Unlike BoW and TF-IDF, which rely on a bag of individual words and treat each word independently, word embeddings capture contextual relationships between words.

They encode semantic meanings, allowing the model to understand synonyms, analogies, and word similarities (e.g., "king" and "queen" are closer in the vector space).

Embeddings are learned from large corpora, which enables them to generalize better and capture nuanced meanings.


Use cases: Word embeddings are particularly effective for tasks like sentiment analysis, machine translation, and named entity recognition (NER), where understanding the semantic meaning of words in context is crucial.

Sources: Word2Vec paper (Mikolov et al., 2013), GloVe paper (Pennington et al., 2014).



---

2. Contextualized Word Embeddings (e.g., BERT, GPT)

How it works: BERT (Bidirectional Encoder Representations from Transformers, Devlin et al., 2018) and models based on GPT (Generative Pretrained Transformers) learn contextualized word representations by considering the surrounding words in both directions. Unlike traditional embeddings, these models produce different representations for a word depending on the context in which it appears.

Advantage over Word2Vec/BoW/TF-IDF:

BERT and similar models are capable of capturing rich semantic and syntactic information within the context of an entire sentence or document, making them far superior for many NLP tasks, especially when words have multiple meanings (e.g., "bank" in "river bank" vs. "financial bank").

These models also handle out-of-vocabulary words better, as they dynamically generate embeddings based on context.

BERT and other transformer-based models are often fine-tuned for specific tasks, resulting in state-of-the-art performance in classification, question answering, and more.


Use cases: BERT and its variants are currently the best-performing models for tasks such as sentiment analysis, text classification, question answering, and machine translation.

Sources: BERT paper (Devlin et al., 2018), GPT-3 Overview (Brown et al., 2020).



---

3. Latent Semantic Analysis (LSA)

How it works: LSA is a technique that reduces the dimensionality of the term-document matrix using Singular Value Decomposition (SVD). This approach captures the latent semantic structure of the text by grouping terms and documents based on shared patterns of co-occurrence.

Advantage over BoW/TF-IDF:

Unlike BoW and TF-IDF, which treat words as independent entities, LSA captures underlying semantic relationships between terms. It represents documents in a reduced-dimensional space, which can enhance generalization by reducing noise from less significant terms.

LSA can handle synonyms and polysemy better than BoW/TF-IDF because it groups similar terms together based on patterns across documents.


Use cases: LSA is useful in information retrieval, document clustering, and topic modeling tasks.

Sources: LSA paper (Deerwester et al., 1990).



---

4. Doc2Vec (Paragraph Vectors)

How it works: Doc2Vec is an extension of Word2Vec, where it generates fixed-length vectors for entire documents, rather than individual words. It learns to represent documents by considering both word sequences and context across the entire document.

Advantage over BoW/TF-IDF:

Unlike BoW or TF-IDF, Doc2Vec captures the meaning of entire documents, which is crucial for tasks like document similarity or document classification.

It considers the overall context and structure of a document, making it more effective for tasks that require understanding the content at the document level.


Use cases: Doc2Vec is widely used in tasks like document classification, document clustering, and information retrieval.

Sources: Doc2Vec paper (Le & Mikolov, 2014).



---

5. Transformer Models (e.g., T5, RoBERTa)

How it works: Models like T5 (Text-to-Text Transfer Transformer) and RoBERTa are based on the transformer architecture but are often fine-tuned for specific tasks like summarization, translation, and text generation. They can generate rich, context-aware embeddings for text and offer better generalization than traditional techniques.

Advantage over BoW/TF-IDF:

These models are pre-trained on large corpora and can generalize better for downstream tasks. They can capture long-range dependencies, making them particularly useful for tasks like text generation and machine translation.

They can also handle multiple languages and work with noisy or informal text more effectively than TF-IDF or BoW.


Use cases: T5 and RoBERTa are widely used in tasks such as text summarization, machine translation, text generation, and sentiment analysis.

Sources: T5 paper (Raffel et al., 2020), RoBERTa paper (Liu et al., 2019).



---

Summary of Advantages

Contextualization: Techniques like BERT and Word2Vec capture semantic meaning and contextual information, leading to better understanding and generalization compared to the simpler, frequency-based methods like BoW and TF-IDF.

Dimensionality reduction: Methods like LSA and Doc2Vec are better at reducing high-dimensional spaces, capturing meaningful patterns in the data.

Task-specific performance: Transformer-based models like T5 and RoBERTa perform exceptionally well across a wide range of NLP tasks due to their pre-training on large corpora.


Each of these methods offers a more sophisticated approach to text representation, providing better performance in NLP tasks, especially when dealing with large-scale, complex datasets.

Stemming vs. Lemmatization: Pros and Cons

Both stemming and lemmatization are techniques in natural language processing (NLP) used to reduce words to their base or root forms. However, they differ in their approach, accuracy, and use cases. Here?s a comparison of the two methods:


---

1. Stemming

Stemming refers to the process of reducing a word to its "stem," which may not necessarily be a valid word itself. The goal is to remove suffixes or prefixes from a word in order to standardize different word forms (e.g., "running," "runner," and "ran" all reduce to "run").

Pros of Stemming:

Speed: Stemming is typically faster because it applies simple, rule-based approaches to remove affixes. This makes it suitable for real-time applications.

Simplicity: Stemming is straightforward to implement and doesn?t require an extensive dictionary or morphological analysis.

Use in Information Retrieval: It's commonly used in search engines or information retrieval systems where the focus is on finding relevant documents rather than understanding precise meanings.


Cons of Stemming:

Over-simplification: Since stemming often removes endings based on a set of rules, it can lead to non-dictionary words that may not make sense or may lose their intended meaning (e.g., "happiest" becomes "happi").

Loss of Information: Stemming may cause a loss of important syntactic or semantic information, especially in complex sentences where word forms matter.

Accuracy Issues: Stemming algorithms like the Porter Stemmer are prone to errors and may produce stems that are not meaningful or linguistically correct.


Sources:

Stemming in NLP: Definition and Techniques

Introduction to Stemming



---

2. Lemmatization

Lemmatization involves reducing a word to its base or dictionary form, known as the lemma, which ensures that the root word is always meaningful. Unlike stemming, lemmatization considers the word's context and part of speech to determine its correct base form.

Pros of Lemmatization:

Accuracy: Lemmatization is more accurate because it ensures that the output is a valid word (e.g., "better" becomes "good," and "running" becomes "run"). It relies on a dictionary and language rules to determine the correct base form.

Context Awareness: By considering the word?s part of speech (e.g., noun, verb), lemmatization preserves the meaning of the word better than stemming.

Semantic Integrity: Lemmatization is better suited for tasks requiring accurate semantic interpretation, like text summarization or sentiment analysis.


Cons of Lemmatization:

Speed: Lemmatization is computationally more expensive than stemming. It requires the use of a dictionary and involves more complex algorithms to resolve ambiguities (e.g., distinguishing between "run" as a verb and "run" as a noun).

Complexity: Lemmatization is more complex to implement than stemming, as it may involve additional preprocessing steps, like part-of-speech tagging, to ensure correct lemmatization.

Requires a Dictionary: Lemmatization is heavily dependent on a lexicon or corpus, which may not always be available for all languages or specialized domains.


Sources:

[Lemmatization vs. Stemming](https://stackabuse.com/lemmatization-vs

### Useful Resources for further reading
1. Stemming and Lemmatization: https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html
2. TF-IDF and BoW : https://www.analyticsvidhya.com/blog/2020/02/quick-introduction-bag-of-words-bow-tf-idf/
3. TF-IDF: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html
